# Makemore Part 3: Activations, Gradients and BatchNorm

Show these things in the Notebook: 

1. Show the histogram of h.view 

2. why we want tan h to not have 1 and -1 as value

3. what is dead neuron: why it will never learn if its abs() is more than 0.99, what to do if a column have all dead neauron

4. add histogram of how to standard deviation of the normal layer when we multiply it: by multiplying to to the inner outer 1 / (row count **0.5). The 1 here will be different for different activation functions, like for tanh its 5/3. for reLu is 1 etc..

5. how and why we found mean and std of a nn before training it for the whole training set instead of finding it on the batch + how we made it from outside the batch to inside chanign per training batch i.e bn mean running, actual and running what is it 

6. what does all of those plot means and how will I use them, each one 

7. 

## Notes

1. **Mean** — center of the distribution. Want `hpreact` mean near 0, or activations are skewed one direction before `tanh` even sees them.

2. **Variance** — average squared spread around the mean.

3. **Std** — `sqrt(variance)`, same info but in real units so you can compare directly to `tanh`'s range of (-1, 1).

**Why it matters:**

- **Low std** → values bunched tightly near the mean → in `hpreact`, most values sit close to 0 → after `tanh`, stuck in the near-linear middle region → weak, underused nonlinearity.
- **High std** → values spread widely, many far from 0 → in `hpreact`, pushes values into the extremes → `tanh` saturates there → dead gradients.
- **"Just right" std** → spread wide enough to use `tanh`'s dynamic range, not so wide that most values sit in the flat saturated zones.

4.

**NOTE 1**: We want our initial training loss close to `-ln(1/num_output_neurons)` — our target initial loss. If it's higher than that, it means we have too much variance: the softmax is producing a very peaked, confident distribution that happens to be confidently wrong.

- **FIX**: scale down the output layer weights (multiply by a small decimal like 0.01) and set bias to 0 initially.

- **CAUTION — don't overdo the same trick on hidden layers**: this fix works for the *output* layer because we want low-confidence logits at init, and there's no activation function after it to worry about. But if you scale weights *feeding into* `tanh` too aggressively, `hpreact` becomes tiny, everything sits near 0, and `tanh` is nearly linear there. Gradients flow fine, but you've thrown away the nonlinearity — the signal becomes too weak to represent anything useful.
  - This is **not** the dead neuron problem — it's the opposite failure mode.
  - **Dead neuron** = weights *too large* → `hpreact` saturates `tanh` at ±1 → `tanh'(x) ≈ 0` → gradient stops flowing → neuron stops learning.
  - **Weak signal** = weights *too small* → `tanh` stays near-linear → nonlinearity wasted, but gradients still flow fine.

**NOTE 2:** Bias is not useful in a layer that feeds into BatchNorm.

-   **Reason:** BatchNorm subtracts the batch mean from every example. Since bias is a constant added equally to every example, subtracting the mean cancels it out entirely — it has zero effect on the output and receives no useful gradient. BatchNorm's own learnable shift (`bnbias` / `beta`) replaces that role, so the layer's bias becomes redundant.